# Foundation Model Training Pipeline
## From Embeddings to Exon Classifier

This notebook walks through the complete foundation model training workflow:

1. **Resource Check** — What can your hardware run?
2. **Generate Embeddings** — Synthetic data for local testing
3. **Train ExonClassifier** — Per-nucleotide exon/intron prediction
4. **Evaluate** — Metrics, training curves, per-gene analysis

We use **synthetic data** so everything runs locally without Evo2 or a GPU.
With real Evo2 embeddings (from a remote GPU), expect AUROC > 0.9 instead of ~0.5.

### Prerequisites
```bash
pip install -e .                    # main project
pip install -e ./foundation_models  # foundation models sub-project
```

### Related Scripts
- `examples/foundation_models/02_synthetic_training_pipeline.py` — CLI version of this notebook
- `examples/foundation_models/05_run_pipeline.py` — Full orchestrator with SkyPilot integration

In [ ]:
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

OUTPUT_DIR = Path("/tmp/fm_notebook/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Output directory: {OUTPUT_DIR}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cpu'}")

## Step 1: Resource Check

Before running expensive operations, check what your hardware can handle.
The resource estimator knows Evo2 model sizes and common GPU profiles.

In [ ]:
from foundation_models.utils.resources import (
    check_current_hardware,
    print_feasibility_report,
)

hw = check_current_hardware()
print(f"Detected: {hw['label']} ({hw['device']}, {hw['memory_gb']} GB)")
print()

# Full feasibility table
print_feasibility_report()

# What about an A40 pod?
print("\n--- Simulating A40 GPU ---\n")
print_feasibility_report(hardware="a40-48gb")

## Step 2: Generate Synthetic Embeddings

Real Evo2 embeddings require loading a 14 GB model. For local development,
we generate **synthetic embeddings** — random vectors with realistic
exon/intron block structure in the labels.

The output format (HDF5 + labels.npz) is identical to what `Evo2Embedder` produces.

In [ ]:
from foundation_models.utils import save_synthetic_embeddings

N_GENES = 8
HIDDEN_DIM = 256  # Real Evo2 7B = 2560, 40B = 5120

hdf5_path, labels_dict = save_synthetic_embeddings(
    output_path=OUTPUT_DIR / "embeddings.h5",
    n_genes=N_GENES,
    hidden_dim=HIDDEN_DIM,
    seed=SEED,
)
labels_path = hdf5_path.with_suffix(".labels.npz")

# Inspect the HDF5 structure
with h5py.File(hdf5_path, "r") as hf:
    print(f"HDF5 file: {hdf5_path}")
    print(f"  hidden_dim: {hf.attrs.get('hidden_dim')}")
    print(f"  model: {hf.attrs.get('model', 'synthetic')}")
    print(f"  genes: {len(hf.keys())}")
    print()
    for gene_id in list(hf.keys())[:5]:
        emb = hf[gene_id]
        lbl = labels_dict[gene_id]
        exon_frac = lbl.mean()
        print(f"  {gene_id}: shape={emb.shape}, exon_fraction={exon_frac:.2f}")

### Data Format

- **HDF5 embeddings**: One dataset per gene, shape `[seq_len, hidden_dim]`, float32
- **Labels NPZ**: One array per gene, shape `[seq_len]`, binary (1 = exon, 0 = intron)
- Labels have realistic block structure: exons 100-500 bp, introns 500-5000 bp

Let's visualize the exon/intron structure of one gene:

In [ ]:
# Pick first gene and plot its exon labels
gene_ids = sorted(labels_dict.keys())
example_gene = gene_ids[0]
example_labels = labels_dict[example_gene]

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.fill_between(range(len(example_labels)), example_labels,
                step="mid", alpha=0.7, color="#3572a5", label="Exon")
ax.set_xlim(0, len(example_labels))
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel("Position (bp)")
ax.set_ylabel("Label")
ax.set_title(f"{example_gene} — Exon/Intron Structure ({len(example_labels):,} bp)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Count exons
transitions = np.diff(example_labels.astype(int))
n_exons = np.sum(transitions == 1) + (1 if example_labels[0] == 1 else 0)
print(f"{example_gene}: {n_exons} exons, {len(example_labels):,} bp, "
      f"exon fraction = {example_labels.mean():.2%}")

## Step 3: Train ExonClassifier

The ExonClassifier is a lightweight model that predicts exon/intron at each
nucleotide position from pre-computed Evo2 embeddings. Available architectures:
- **linear**: Single linear layer (baseline)
- **mlp**: Multi-layer perceptron (default)
- **cnn**: 1D convolution (captures local context)
- **lstm**: Bidirectional LSTM (captures long-range context)

Training requires fixed-size windows, so we use `window_embeddings()` to
slide a window over variable-length gene embeddings.

In [ ]:
from foundation_models.evo2 import ExonClassifier
from foundation_models.utils.chunking import window_embeddings

WINDOW_SIZE = 512
STEP_SIZE = 256
ARCHITECTURE = "mlp"

# Create windows from all genes
emb_file = h5py.File(hdf5_path, "r")
labels_data = np.load(labels_path)
gene_ids = list(emb_file.keys())

# Split: last 20% of genes for validation
n_val = max(1, int(len(gene_ids) * 0.2))
val_genes = set(gene_ids[-n_val:])

train_windows, val_windows = [], []
for gene_id in gene_ids:
    emb = emb_file[gene_id][:]
    lbl = labels_data[gene_id]
    min_len = min(len(emb), len(lbl))

    windows = window_embeddings(
        embeddings=emb[:min_len], labels=lbl[:min_len],
        gene_id=gene_id, window_size=WINDOW_SIZE, step_size=STEP_SIZE,
    )
    if gene_id in val_genes:
        val_windows.extend(windows)
    else:
        train_windows.extend(windows)

emb_file.close()

print(f"Train windows: {len(train_windows)}")
print(f"Val windows:   {len(val_windows)}")
print(f"Window size:   {WINDOW_SIZE} bp, step: {STEP_SIZE} bp")
print(f"Train genes:   {len(gene_ids) - n_val}, val genes: {n_val}")

# Stack into tensors
train_emb = torch.tensor(np.stack([w[0] for w in train_windows]), dtype=torch.float32)
train_lbl = torch.tensor(np.stack([w[1] for w in train_windows]), dtype=torch.float32)
val_emb = torch.tensor(np.stack([w[0] for w in val_windows]), dtype=torch.float32)
val_lbl = torch.tensor(np.stack([w[1] for w in val_windows]), dtype=torch.float32)

print(f"\nTrain tensor: {list(train_emb.shape)}")
print(f"Val tensor:   {list(val_emb.shape)}")
print(f"Memory: {(train_emb.nelement() + val_emb.nelement()) * 4 / 1e6:.0f} MB")

In [ ]:
# Create and train the classifier
classifier = ExonClassifier(
    input_dim=HIDDEN_DIM,
    hidden_dim=min(256, HIDDEN_DIM),
    num_layers=2,
    architecture=ARCHITECTURE,
    dropout=0.1,
)
n_params = sum(p.numel() for p in classifier.parameters())
print(f"Architecture: {ARCHITECTURE}")
print(f"Parameters:   {n_params:,}")
print()

t0 = time.time()
history = classifier.fit(
    train_embeddings=train_emb,
    train_labels=train_lbl,
    val_embeddings=val_emb,
    val_labels=val_lbl,
    epochs=25,
    batch_size=32,
    lr=1e-3,
    device="cpu",
    verbose=True,
    checkpoint_dir=str(OUTPUT_DIR / "checkpoints"),
    patience=10,
    lr_schedule=True,
)
elapsed = time.time() - t0
print(f"\nTraining time: {elapsed:.1f}s")
print(f"Best epoch: {history['best_epoch'] + 1}, AUROC: {history['best_val_auroc']:.4f}")
print(f"Early stopped: {history['stopped_early']}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history["train_loss"]) + 1)

# Loss
ax1.plot(epochs, history["train_loss"], label="Train", color="#3572a5")
ax1.plot(epochs, history["val_loss"], label="Val", color="#e74c3c")
ax1.axvline(history["best_epoch"] + 1, color="gray", linestyle="--",
            alpha=0.5, label=f"Best (epoch {history['best_epoch'] + 1})")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.legend()

# AUROC
ax2.plot(epochs, history["val_auroc"], label="Val AUROC", color="#27ae60", linewidth=2)
ax2.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="Random baseline")
ax2.axvline(history["best_epoch"] + 1, color="gray", linestyle="--",
            alpha=0.5, label=f"Best ({history['best_val_auroc']:.4f})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("AUROC")
ax2.set_title("Validation AUROC")
ax2.set_ylim(0.4, 0.6)
ax2.legend()

plt.suptitle(f"ExonClassifier ({ARCHITECTURE}) — Synthetic Data", fontsize=13)
plt.tight_layout()
plt.show()

print("Note: AUROC ~0.5 is expected — random embeddings carry no exon signal.")
print("With real Evo2 embeddings, expect AUROC > 0.9.")

## Step 4: Evaluate

Evaluate the trained classifier on the validation set. We compute:
- **Overall metrics**: AUROC, AUPRC, F1, precision, recall
- **Per-gene AUROC**: How well the model works for each gene individually

In [ ]:
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)

# Overall predictions
probs = classifier.predict(val_emb).flatten()
labels_flat = val_lbl.numpy().flatten()
preds = (probs >= 0.5).astype(int)

try:
    auroc = roc_auc_score(labels_flat, probs)
except ValueError:
    auroc = float("nan")
try:
    auprc = average_precision_score(labels_flat, probs)
except ValueError:
    auprc = float("nan")

print("Overall Metrics:")
print(f"  AUROC:     {auroc:.4f}")
print(f"  AUPRC:     {auprc:.4f}")
print(f"  Accuracy:  {accuracy_score(labels_flat, preds):.4f}")
print(f"  F1:        {f1_score(labels_flat, preds, zero_division=0):.4f}")
print(f"  Precision: {precision_score(labels_flat, preds, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(labels_flat, preds, zero_division=0):.4f}")

In [ ]:
# Per-gene AUROC
emb_file = h5py.File(hdf5_path, "r")
labels_data = np.load(labels_path)

per_gene = {}
for gene_id in val_genes:
    emb = emb_file[gene_id][:]
    lbl = labels_data[gene_id]
    min_len = min(len(emb), len(lbl))

    windows = window_embeddings(
        embeddings=emb[:min_len], labels=lbl[:min_len],
        gene_id=gene_id, window_size=WINDOW_SIZE, step_size=STEP_SIZE,
    )
    if not windows:
        continue

    win_emb = torch.tensor(np.stack([w[0] for w in windows]), dtype=torch.float32)
    win_lbl = np.concatenate([w[1] for w in windows])
    win_probs = classifier.predict(win_emb).flatten()

    try:
        gene_auroc = roc_auc_score(win_lbl, win_probs)
    except ValueError:
        gene_auroc = float("nan")
    per_gene[gene_id] = gene_auroc

emb_file.close()

# Bar chart
if per_gene:
    sorted_genes = sorted(per_gene.items(), key=lambda x: x[1], reverse=True)
    names = [g[0] for g in sorted_genes]
    aurocs = [g[1] for g in sorted_genes]

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#27ae60" if a > 0.5 else "#e74c3c" for a in aurocs]
    ax.barh(names, aurocs, color=colors, edgecolor="white")
    ax.axvline(0.5, color="gray", linestyle="--", alpha=0.5, label="Random")
    ax.set_xlabel("AUROC")
    ax.set_xlim(0, 1)
    ax.set_title("Per-Gene AUROC (Validation Set)")
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No per-gene results (validation set may be too small).")

## Remote GPU Workflow

For real Evo2 embeddings, you need a GPU. The workflow:

```bash
# 1. Check resources
python examples/foundation_models/01_resource_check.py --hardware a40-48gb

# 2. Launch embedding extraction on a remote A40 pod
sky launch foundation_models/configs/skypilot/extract_embeddings_a40.yaml

# 3. Download results
sky rsync extract-emb-7b /workspace/output/ ./output/embeddings/

# 4. Tear down
sky down extract-emb-7b -y

# 5. Train locally (classifier is tiny)
python examples/foundation_models/04_train_and_evaluate.py \
    --embeddings ./output/embeddings/embeddings.h5 \
    --labels ./output/embeddings/embeddings.labels.npz \
    --output ./output/checkpoints/
```

Or use the pipeline orchestrator to do all steps in one command:

```bash
# Dry-run: see what would happen
python examples/foundation_models/05_run_pipeline.py \
    --model-size 7b --gpu a40 --chromosomes 22

# Execute: actually launch jobs
python examples/foundation_models/05_run_pipeline.py \
    --execute --model-size 7b --gpu a40 --chromosomes 22 --train-local
```

Below we show what the orchestrator generates programmatically:

In [ ]:
import yaml

# This mirrors what 05_run_pipeline.py generates
GPU_SPECS = {
    "a40":  {"accelerator": "A40:1",       "hourly_rate": 0.39, "label": "NVIDIA A40 48 GB"},
    "a100": {"accelerator": "A100-80GB:1",  "hourly_rate": 1.64, "label": "NVIDIA A100 80 GB"},
    "h100": {"accelerator": "H100-80GB:1",  "hourly_rate": 3.29, "label": "NVIDIA H100 80 GB"},
}

gpu = "a40"
model_size = "7b"
chromosomes = [22]

config = {
    "name": f"fm-{model_size}-chr{chromosomes[0]}",
    "resources": {
        "accelerators": GPU_SPECS[gpu]["accelerator"],
        "cloud": "runpod",
    },
    "file_mounts": {
        "/workspace/data": {"source": "./data/mane/GRCh38/", "mode": "COPY"},
    },
    "setup": "pip install -e .\npip install -e ./foundation_models",
    "run": (
        f"python foundation_models/examples/02_extract_embeddings.py \\"
        f"\n  --data-dir /workspace/data/ \\"
        f"\n  --chromosomes {' '.join(str(c) for c in chromosomes)} \\"
        f"\n  --output /workspace/output/embeddings.h5"
    ),
}

print("Generated SkyPilot config:")
print("=" * 40)
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

rate = GPU_SPECS[gpu]["hourly_rate"]
print(f"Estimated cost: ~${rate * 0.5:.2f} ({GPU_SPECS[gpu]['label']} @ ${rate:.2f}/hr for ~30 min)")

## Summary

### What We Built

1. **Resource check** — feasibility table for current and target hardware
2. **Synthetic embeddings** — HDF5 + labels.npz with realistic exon/intron structure
3. **ExonClassifier** — trained with early stopping and LR scheduling
4. **Evaluation** — overall and per-gene AUROC, training curves

### Key Takeaways

- AUROC ~0.5 on synthetic data is expected (random embeddings have no exon signal)
- With real Evo2 embeddings, expect AUROC > 0.9
- The classifier is tiny (~130K params for MLP) — training is cheap
- The expensive step is embedding extraction (loading Evo2 7B/40B)
- Decoupling embedding extraction from training enables experimentation

### Next Steps

- Try different architectures: rerun with `ARCHITECTURE = "cnn"` or `"lstm"`
- Extract real embeddings: `examples/foundation_models/03_embedding_extraction.py`
- Full pipeline: `examples/foundation_models/05_run_pipeline.py`
- Sub-project docs: `foundation_models/docs/`